L'objectif de ce notebook est de construire un modèle de recommandation content-based enrichi, en combinant plusieurs sources d'information sur chaque film : les genres (MovieLens), le synopsis, le réalisateur et les acteurs principaux (récupérés via l'API TMDb dans le notebook précédent). Contrairement au premier modèle content-based (notebook n°2), qui se limitait aux 19 genres et obtenait des résultats très faibles lors de l'évaluation par Precision@k/Recall@k, ce modèle vise à capturer des similarités plus fines entre les films en tenant compte du contenu narratif (synopsis) et de l'équipe artistique (acteurs, réalisateur). Chaque source d'information sera transformée en vecteur numérique séparément, puis combinée en un vecteur unique par film à l'aide d'une pondération que nous choisirons et justifierons, avant de recalculer la similarité entre films sur cette nouvelle base enrichie.

Étape 1 : Charger le fichier enrichi 

In [20]:
import pandas as pd
import numpy as np

films = pd.read_csv('/Users/nazmanazirhussain/Desktop/RecommandationFilm/data/enrichi/films_complet.csv')

print("Dimensions :", films.shape)
films.head()

Dimensions : (9742, 9)


,movieId,title,genres,annee,titre,genres_list,synopsis,realisateur,acteurs
0,1,Toy Story (1995),Adventure|Animation|Children|Comedy|Fantasy,1995,Toy Story,"['Adventure', 'Animation', 'Children', 'Comedy...",Dans un monde où les jouets vivent leur vie qu...,John Lasseter,"['Tom Hanks', 'Tim Allen', 'Don Rickles']"
1,2,Jumanji (1995),Adventure|Children|Fantasy,1995,Jumanji,"['Adventure', 'Children', 'Fantasy']","Lors d'une partie de Jumanji, un jeu très anci...",Joe Johnston,"['Robin Williams', 'Kirsten Dunst', 'Bradley P..."
2,3,Grumpier Old Men (1995),Comedy|Romance,1995,Grumpier Old Men,"['Comedy', 'Romance']","Après le mariage de John et d'Ariel, Max se re...",Howard Deutch,"['Walter Matthau', 'Jack Lemmon', 'Ann-Margret']"
3,4,Waiting to Exhale (1995),Comedy|Drama|Romance,1995,Waiting to Exhale,"['Comedy', 'Drama', 'Romance']",L'amitié de quatre femmes qui tentent de surmo...,Forest Whitaker,"['Whitney Houston', 'Angela Bassett', 'Loretta..."
4,5,Father of the Bride Part II (1995),Comedy,1995,Father of the Bride Part II,['Comedy'],George Banks et sa petite famille sont de reto...,Charles Shyer,"['Steve Martin', 'Diane Keaton', 'Martin Short']"


Étape 2 : Préparer les synopsis pour le TF-IDF

Avant de pouvoir transformer les synopsis en vecteurs numériques, il faut les nettoyer. 730 films possèdent une valeur manquante (NaN) dans la colonne synopsis : 121 films n'ayant pas pu être associés à un identifiant TMDb valide, et 609 films supplémentaires bien référencés sur TMDb mais sans description textuelle disponible. Nous remplaçons ces valeurs manquantes par une chaîne de caractères vide (''), ce qui revient à considérer que ces films n'ont aucune information textuelle disponible, sans pour autant les exclure du reste de l'analyse.

In [21]:
# On remplace les synopsis manquants par une chaîne vide, pour éviter les erreurs de calcul
films['synopsis'] = films['synopsis'].fillna('')

# On vérifie qu'il n'y a plus de valeurs manquantes
print("Films sans synopsis restants :", (films['synopsis'] == '').sum())

Films sans synopsis restants : 730


Étape 3 : Construire la matrice TF-IDF sur les synopsis 

In [22]:
from sklearn.feature_extraction.text import TfidfVectorizer

# Scikit-learn ne connaît pas nativement les mots vides français (contrairement à l'anglais),
# on définit donc notre propre liste des mots les plus fréquents et peu informatifs en français

mots_liste = [
    'le', 'la', 'les', 'un', 'une', 'des', 'de', 'du', 'et', 'ou', 'mais',
    'où', 'donc', 'or', 'ni', 'car', 'que', 'qui', 'quoi', 'dont', 'ce',
    'cette', 'ces', 'cet', 'son', 'sa', 'ses', 'son', 'leur', 'leurs',
    'mon', 'ma', 'mes', 'ton', 'ta', 'tes', 'notre', 'nos', 'votre', 'vos',
    'il', 'elle', 'ils', 'elles', 'je', 'tu', 'nous', 'vous', 'on',
    'est', 'sont', 'était', 'être', 'avoir', 'ont', 'a', 'as', 'ai',
    'dans', 'sur', 'sous', 'avec', 'sans', 'pour', 'par', 'en', 'au', 'aux',
    'à', 'se', 'sa', 'lui', 'leur', 'y', 'là', 'pas', 'plus', 'très',
    'tout', 'tous', 'toute', 'toutes', 'comme', 'quand', 'si', 'ne'
]

tfidf_synopsis = TfidfVectorizer(stop_words=mots_liste, min_df=2)
matrice_tfidf_synopsis = tfidf_synopsis.fit_transform(films['synopsis'])

print("Dimensions de la matrice TF-IDF (synopsis) :", matrice_tfidf_synopsis.shape)
print("Nombre de mots uniques retenus :", len(tfidf_synopsis.get_feature_names_out()))

Dimensions de la matrice TF-IDF (synopsis) : (9742, 20486)
Nombre de mots uniques retenus : 20486


Étape 4 : Test 

In [23]:
from sklearn.metrics.pairwise import cosine_similarity

# On calcule la similarité entre tous les films, cette fois basée sur les synopsis
similarite_synopsis = cosine_similarity(matrice_tfidf_synopsis, matrice_tfidf_synopsis)

print("Dimensions de la matrice de similarité :", similarite_synopsis.shape)

Dimensions de la matrice de similarité : (9742, 9742)


Étape 5 : Fonction de test rapide 

In [24]:
def films_similaires_synopsis(titre_film, n=10):
    idx = films[films['titre'] == titre_film].index[0]
    scores = list(enumerate(similarite_synopsis[idx]))
    scores = sorted(scores, key=lambda x: x[1], reverse=True)[1:n+1]
    indices = [i[0] for i in scores]
    return films.iloc[indices][['titre', 'genres', 'annee']]

In [25]:
# Testons notre fonction avec le film "Toy Story"

films_similaires_synopsis('Toy Story')

,titre,genres,annee
7355,Toy Story 3,Adventure|Animation|Children|Comedy|Fantasy|IMAX,2010
2355,Toy Story 2,Adventure|Animation|Children|Comedy|Fantasy,1999
6224,Lady in the Water,Drama|Fantasy|Mystery,2006
2329,Babes in Toyland,Children|Comedy|Fantasy|Musical,1934
1401,Small Soldiers,Animation|Children|Fantasy|War,1998
9207,Welcome to Happiness,Comedy|Drama|Fantasy,2015
3109,Of Mice and Men,Drama,1939
4890,Hidalgo,Adventure|Drama,2004
4209,Dragon Lord (a.k.a. Dragon Strike) (Long Xiao Ye),Action,1982
2854,Tampopo,Comedy,1985


Commentaire : 

Un test sur Toy Story révèle un apport clair du synopsis par rapport aux genres seuls : les deux suites de la saga (Toy Story 2 et 3) remontent naturellement en tête des recommandations, ce que le modèle basé uniquement sur les genres ne permettait pas de distinguer parmi les nombreux films d'animation familiale partageant les mêmes genres. Le synopsis capture ainsi une dimension narrative et thématique que les genres, trop généraux, ne peuvent pas représenter. On observe cependant aussi quelques résultats moins évidents (Tampopo, Dragon Lord), probablement liés à des similarités de vocabulaire non thématiquement pertinentes, une limite courante des approches TF-IDF sur du texte libre.

Étape 6 : Encoder les acteurs et le réalisateur

Contrairement au synopsis (texte libre) ou aux genres (déjà séparés par |), les acteurs et le réalisateur ont chacun leurs particularités à gérer avant de pouvoir les transformer en vecteurs.

1° Remettre la colonne "acteurs" sous forme de vraie liste 

Quand on a sauvegardé le CSV, nos listes d'acteurs ont été converties en texte simple. Il faut les reconvertir en vraies listes Python avant de pouvoir les utiliser. 

In [26]:
import ast

# La colonne acteurs a été sauvegardée comme du texte (ex: "['Tom Hanks', 'Tim Allen']")

# On la reconvertit en vraie liste Python avec ast.literal_eval

# Pour les films sans acteurs (NaN), on renvoie une liste vide
def convertir_en_liste(valeur):
    if pd.isnull(valeur):
        return []
    return ast.literal_eval(valeur)

films['acteurs_liste'] = films['acteurs'].apply(convertir_en_liste)

films[['titre', 'acteurs', 'acteurs_liste']].head()

,titre,acteurs,acteurs_liste
0,Toy Story,"['Tom Hanks', 'Tim Allen', 'Don Rickles']","[Tom Hanks, Tim Allen, Don Rickles]"
1,Jumanji,"['Robin Williams', 'Kirsten Dunst', 'Bradley P...","[Robin Williams, Kirsten Dunst, Bradley Pierce]"
2,Grumpier Old Men,"['Walter Matthau', 'Jack Lemmon', 'Ann-Margret']","[Walter Matthau, Jack Lemmon, Ann-Margret]"
3,Waiting to Exhale,"['Whitney Houston', 'Angela Bassett', 'Loretta...","[Whitney Houston, Angela Bassett, Loretta Devine]"
4,Father of the Bride Part II,"['Steve Martin', 'Diane Keaton', 'Martin Short']","[Steve Martin, Diane Keaton, Martin Short]"


2° Préparer une colonne texte pour les acteurs (comme on l'avait fait pour les genres)

In [27]:
# On transforme chaque liste d'acteurs en une chaîne de caractères, séparée par des espaces

# On remplace les espaces internes aux noms par un underscore, pour que "Tom Hanks" reste un seul mot

# (sinon "Tom" et "Hanks" seraient traités comme deux informations séparées, ce qui n'a pas de sens)
def preparer_texte_acteurs(liste_acteurs):
    noms_sans_espace = [nom.replace(' ', '_') for nom in liste_acteurs]
    return ' '.join(noms_sans_espace)

films['acteurs_str'] = films['acteurs_liste'].apply(preparer_texte_acteurs)

films[['titre', 'acteurs_liste', 'acteurs_str']].head()

,titre,acteurs_liste,acteurs_str
0,Toy Story,"[Tom Hanks, Tim Allen, Don Rickles]",Tom_Hanks Tim_Allen Don_Rickles
1,Jumanji,"[Robin Williams, Kirsten Dunst, Bradley Pierce]",Robin_Williams Kirsten_Dunst Bradley_Pierce
2,Grumpier Old Men,"[Walter Matthau, Jack Lemmon, Ann-Margret]",Walter_Matthau Jack_Lemmon Ann-Margret
3,Waiting to Exhale,"[Whitney Houston, Angela Bassett, Loretta Devine]",Whitney_Houston Angela_Bassett Loretta_Devine
4,Father of the Bride Part II,"[Steve Martin, Diane Keaton, Martin Short]",Steve_Martin Diane_Keaton Martin_Short


3° Préparer une colonne texte pour le réalisateur 

In [28]:
# On fait la même transformation pour le réalisateur : remplacer les espaces par des underscores

# et gérer les valeurs manquantes (4 films sans réalisateur identifié)
def preparer_texte_realisateur(nom):
    if pd.isnull(nom):
        return ''
    return nom.replace(' ', '_')

films['realisateur_str'] = films['realisateur'].apply(preparer_texte_realisateur)

films[['titre', 'realisateur', 'realisateur_str']].head()

,titre,realisateur,realisateur_str
0,Toy Story,John Lasseter,John_Lasseter
1,Jumanji,Joe Johnston,Joe_Johnston
2,Grumpier Old Men,Howard Deutch,Howard_Deutch
3,Waiting to Exhale,Forest Whitaker,Forest_Whitaker
4,Father of the Bride Part II,Charles Shyer,Charles_Shyer


4° Construire les matrices TF-IDF pour acteurs et réalisateur

In [29]:
# On utilise le même principe que pour les genres : découper uniquement sur les espaces

# (token_pattern=r'[^\s]+'), puisque chaque nom est déjà un seul "mot" grâce aux underscores

tfidf_acteurs = TfidfVectorizer(token_pattern=r'[^\s]+')
matrice_tfidf_acteurs = tfidf_acteurs.fit_transform(films['acteurs_str'])

tfidf_realisateur = TfidfVectorizer(token_pattern=r'[^\s]+')
matrice_tfidf_realisateur = tfidf_realisateur.fit_transform(films['realisateur_str'])

print("Dimensions matrice acteurs :", matrice_tfidf_acteurs.shape)
print("Dimensions matrice réalisateur :", matrice_tfidf_realisateur.shape)

Dimensions matrice acteurs : (9742, 11077)
Dimensions matrice réalisateur : (9742, 3978)


7° Combiner les 4 matrices en un seul vecteur pondéré

Objectif : Nous disposons désormais de quatre représentations vectorielles distinctes pour chaque film : les genres (19 dimensions), le synopsis (20 486 dimensions), les acteurs principaux (11 077 dimensions) et le réalisateur (3 978 dimensions). L'objectif de cette étape est de combiner ces quatre sources en un seul vecteur par film, afin de calculer une similarité globale qui tienne compte de l'ensemble de ces critères, plutôt que d'un seul à la fois. Chaque source étant multipliée par un poids avant d'être assemblée, l'importance relative de chaque type d'information dans le calcul final de similarité peut être ajustée et justifiée.

Choix et justification des poids : 

Nous attribuons les poids suivants : genres (0,35), acteurs (0,30), synopsis (0,25), réalisateur (0,10). Les genres reçoivent le poids le plus élevé car il s'agit du signal le plus stable et le moins bruité : un thriller reste un thriller, indépendamment des autres films dans lesquels ses acteurs ou son réalisateur ont pu jouer. Les acteurs suivent avec un poids important, en cohérence avec l'idée que deux films partageant les mêmes acteurs principaux ont souvent un ton ou un style reconnaissable pour le spectateur. Le synopsis conserve un poids significatif, car il apporte une nuance thématique que les genres seuls ne peuvent pas capturer, mais reste légèrement inférieur aux deux premiers critères car c'est la source la plus sujette au bruit (mots de vocabulaire non pertinents). Enfin, le réalisateur reçoit le poids le plus faible : bien qu'il puisse représenter un vrai style artistique, c'est un critère plus subtil que la majorité des spectateurs ne perçoit pas consciemment dans ses préférences.

1° Recharger la matrice des genres 

In [30]:
import pickle

# La matrice TF-IDF des genres a été créée et sauvegardée dans le notebook 2.
# Chaque notebook ayant sa propre mémoire, on doit la recharger ici.
with open('/Users/nazmanazirhussain/Desktop/RecommandationFilm/models/matrice_tfidf.pkl', 'rb') as fichier:
    matrice_tfidf_genres = pickle.load(fichier)

# Vérification de cohérence : le nombre de films doit correspondre partout
print("Dimensions matrice genres :", matrice_tfidf_genres.shape)
print("Nombre de films dans le tableau films :", films.shape[0])

Dimensions matrice genres : (9742, 19)
Nombre de films dans le tableau films : 9742


2° Définir les poids

In [31]:
# Poids attribués à chaque source d'information (justification en cellule Markdown ci-dessus)
poids_genres = 0.35
poids_acteurs = 0.30
poids_synopsis = 0.25
poids_realisateur = 0.10

print("Somme des poids :", round(poids_genres + poids_acteurs + poids_synopsis + poids_realisateur, 2))

Somme des poids : 1.0


3° Combiner les 4 matrices pondérées 

In [32]:
from scipy.sparse import hstack

# On combine les 4 matrices côte à côte (hstack = horizontal stack), chacune multipliée par son poids avant d'être assemblée

matrice_combinee = hstack([
    matrice_tfidf_genres * poids_genres,
    matrice_tfidf_acteurs * poids_acteurs,
    matrice_tfidf_synopsis * poids_synopsis,
    matrice_tfidf_realisateur * poids_realisateur
])

print("Dimensions de la matrice combinée :", matrice_combinee.shape)

Dimensions de la matrice combinée : (9742, 35560)


4° Calculer la similarité entre films 

In [33]:
from sklearn.metrics.pairwise import cosine_similarity

similarite_combinee = cosine_similarity(matrice_combinee, matrice_combinee)

print("Dimensions de la matrice de similarité combinée :", similarite_combinee.shape)

Dimensions de la matrice de similarité combinée : (9742, 9742)


5° Fonction de recommanadation 

In [34]:
def films_similaires_combine(titre_film, n=10):
    idx = films[films['titre'] == titre_film].index[0]
    scores = list(enumerate(similarite_combinee[idx]))
    scores = sorted(scores, key=lambda x: x[1], reverse=True)[1:n+1]
    indices = [i[0] for i in scores]
    return films.iloc[indices][['titre', 'genres', 'annee', 'realisateur']]

In [35]:
# Tester notre fonction avec le film "Toy Story"

films_similaires_combine('Toy Story')

,titre,genres,annee,realisateur
2355,Toy Story 2,Adventure|Animation|Children|Comedy|Fantasy,1999,John Lasseter
7355,Toy Story 3,Adventure|Animation|Children|Comedy|Fantasy|IMAX,2010,Lee Unkrich
9565,Gulliver's Travels,Adventure|Children|Fantasy,1996,NaN
3680,Escaflowne: The Movie (Escaflowne),Action|Adventure|Animation|Drama|Fantasy,2000,NaN
8988,Afro Samurai,Action|Adventure|Animation|Drama|Fantasy,2007,NaN
9167,North Pole: Open For Christmas,Children|Fantasy,2015,NaN
5627,"10th Kingdom, The",Adventure|Comedy|Fantasy,2000,NaN
6163,"Shaggy Dog, The",Adventure|Children|Comedy|Fantasy,2006,Brian Robbins
9536,Last Year's Snow Was Falling,Animation|Children|Comedy|Fantasy,1983,Александр Татарский
9560,Wow! A Talking Fish!,Animation|Children|Comedy|Fantasy,1983,Ռոբերտ Սահակյանց


Commentaire : Le modèle combiné (genres + acteurs + synopsis + réalisateur) place les deux suites de la franchise (Toy Story 2 et Toy Story 3) en première et deuxième position, avec les bons réalisateurs identifiés (John Lasseter et Lee Unkrich). Ce résultat confirme l'apport de la combinaison de plusieurs sources par rapport aux modèles utilisant une seule information à la fois. On observe également que des films dont le réalisateur n'a pas pu être identifié via l'API TMDb (valeur NaN) restent malgré tout recommandés, ce qui est cohérent avec la pondération choisie : le réalisateur, avec un poids de seulement 0,10, n'empêche pas un film d'être recommandé sur la base des trois autres critères.

In [37]:
# Test n°2 

films_similaires_combine('Spider-Man')


,titre,genres,annee,realisateur
6470,Spider-Man 3,Action|Adventure|Sci-Fi|Thriller|IMAX,2007,Sam Raimi
6894,Farscape: The Peacekeeper Wars,Action|Adventure|Sci-Fi,2004,NaN
8812,Power/Rangers,Action|Adventure|Sci-Fi,2015,NaN
2141,Saturn 3,Adventure|Sci-Fi|Thriller,1980,NaN
5260,Spider-Man 2,Action|Adventure|Sci-Fi|IMAX,2004,Sam Raimi
3741,Ffolkes,Action|Adventure|Thriller,1979,NaN
7063,Raiders of the Lost Ark: The Adaptation,Action|Adventure|Thriller,1989,NaN
9534,T2 3-D: Battle Across Time,Action|Sci-Fi,1996,NaN
8301,"Day of the Doctor, The",Adventure|Drama|Sci-Fi,2013,NaN
5949,Stealth,Action|Adventure|Sci-Fi|Thriller,2005,Rob Cohen


Commentaire : Un second test avec Spider-Man confirme la cohérence du modèle combiné : les deux autres volets de la trilogie originale (Spider-Man 2 et 3) apparaissent en tête des recommandations, avec le réalisateur correctement identifié (Sam Raimi) pour les deux. Les autres films recommandés restent cohérents avec l'univers action/aventure/science-fiction du film de référence, confirmant que le modèle capture efficacement des franchises et des thématiques proches, même en l'absence d'une catégorie de genre spécifique aux films de super-héros dans les données MovieLens.

Étape 8 : Évaluer le modèle combiné avec Precision@k / Recall@k

1° Charger les notes et refaire le même split entrainement/test

In [38]:
notes = pd.read_csv('/Users/nazmanazirhussain/Desktop/RecommandationFilm/data/nettoye/ratings_clean.csv')

from sklearn.model_selection import train_test_split

# On reprend exactement le même split que dans le notebook 3 (même random_state) pour que la comparaison entre modèles reste cohérente
notes_entrainement, notes_test = train_test_split(notes, test_size=0.2, random_state=42)

print("Notes d'entraînement :", notes_entrainement.shape)
print("Notes de test :", notes_test.shape)

Notes d'entraînement : (80668, 5)
Notes de test : (20168, 5)


2° Préparer un dictionnaire de correspondance movieId --> position

In [39]:
# La matrice combinée est indexée dans le même ordre que le tableau films
position_film_combine = {movie_id: i for i, movie_id in enumerate(films['movieId'])}

3° Fonction de recommandation basée sur le profil moyen 

In [40]:
def recommander_films_combine(id_utilisateur, k=10):
    notes_utilisateur = notes_entrainement[notes_entrainement['userId'] == id_utilisateur]
    films_notes_utilisateur = notes_utilisateur['movieId'].tolist()
    
    indices_notes = [position_film_combine[f] for f in films_notes_utilisateur if f in position_film_combine]
    
    if len(indices_notes) == 0:
        return []
    
    # On calcule le profil moyen de l'utilisateur sur la matrice combinée
    profil_utilisateur = matrice_combinee[indices_notes].mean(axis=0)
    profil_utilisateur = np.asarray(profil_utilisateur)
    
    scores = cosine_similarity(profil_utilisateur, matrice_combinee)[0]
    
    for idx in indices_notes:
        scores[idx] = -1
    
    meilleurs_indices = np.argsort(scores)[::-1][:k]
    
    return [films['movieId'].iloc[i] for i in meilleurs_indices]

4° Fonction Precision@k / Recall@k pour ce modèle

In [41]:
def calculer_precision_recall_combine(id_utilisateur, k=10, seuil_appreciation=4.0):
    films_recommandes = recommander_films_combine(id_utilisateur, k=k)
    
    notes_test_utilisateur = notes_test[notes_test['userId'] == id_utilisateur]
    films_aimes = notes_test_utilisateur[notes_test_utilisateur['rating'] >= seuil_appreciation]['movieId'].tolist()
    
    if len(films_aimes) == 0 or len(films_recommandes) == 0:
        return None, None
    
    nb_corrects = len(set(films_recommandes) & set(films_aimes))
    
    precision = nb_corrects / k
    recall = nb_corrects / len(films_aimes)
    
    return precision, recall

5° Tester sur un échantillon de 100 utilisateurs

In [42]:
np.random.seed(42)  # pour rendre l'échantillon reproductible
utilisateurs_test = notes_test['userId'].unique()
echantillon_utilisateurs = np.random.choice(utilisateurs_test, size=100, replace=False)

precisions_combine = []
recalls_combine = []

for id_utilisateur in echantillon_utilisateurs:
    precision, recall = calculer_precision_recall_combine(id_utilisateur, k=10)
    if precision is not None:
        precisions_combine.append(precision)
        recalls_combine.append(recall)

precision_moyenne_combine = np.mean(precisions_combine)
recall_moyen_combine = np.mean(recalls_combine)

print(f"Precision@10 moyenne (combiné) sur {len(precisions_combine)} utilisateurs : {precision_moyenne_combine:.3f}")
print(f"Recall@10 moyen (combiné) sur {len(precisions_combine)} utilisateurs : {recall_moyen_combine:.3f}")

Precision@10 moyenne (combiné) sur 97 utilisateurs : 0.002
Recall@10 moyen (combiné) sur 97 utilisateurs : 0.002


Commentaire : Contrairement à l'hypothèse initiale selon laquelle l'enrichissement du contenu (synopsis, acteurs, réalisateur) améliorerait significativement les performances du modèle content-based, l'évaluation quantitative montre des résultats quasiment identiques au modèle basé sur les genres seuls (Precision@10 de 0,002 contre 0,003). Ce résultat, bien que surprenant au premier abord, révèle une limite plus fondamentale que le simple manque de richesse des données : la méthode de construction du profil utilisateur par moyenne des vecteurs de films notés dilue les préférences individuelles, quelle que soit la richesse du contenu représenté. Le filtrage collaboratif, qui exploite directement les comportements réels de centaines d'utilisateurs plutôt qu'une similarité de contenu, conserve un avantage net et confirme sa pertinence comme modèle principal pour l'application finale. Ce résultat constitue une découverte méthodologique intéressante : l'enrichissement du contenu améliore la cohérence qualitative des recommandations pour un film isolé (comme observé avec Toy Story et Spider-Man), mais ne suffit pas à résoudre les limites structurelles de l'approche par profil moyen pour prédire les préférences réelles d'un utilisateur.

Étape 9 : Tester le modèle dans les conditions réelles de l'application (peu de films sélectionnés)

Objectif : Jusqu'ici, on a testé notre modèle en imaginant des utilisateurs qui ont noté beaucoup de films (parfois plus de 100). Mais ce n'est pas ce que fera un vrai utilisateur de l'application : il choisira seulement entre 1 et 4 films qu'il aime, pas plus. Cette étape teste donc le modèle dans les vraies conditions de l'application : on simule un utilisateur qui ne choisit que 4 films (comme le fera un vrai utilisateur), on calcule les recommandations à partir de ces 4 films uniquement, puis on vérifie si les films recommandés lui plaisent vraiment (en comparant avec d'autres films qu'il a aimés, mis de côté pour le test). L'objectif est de savoir si le modèle enrichi, qui avait donné un mauvais score avec beaucoup de films, fonctionne mieux dans ce cas plus réaliste et plus proche de notre projet.

Etude de cas : Imaginez Camille qui choisit 4 films qu'elle aime vraiment : In the Mood for Love, Her, Lost in Translation, Eternal Sunshine of the Spotless Mind. Le test qu'on va faire, c'est exactement ça, mais fait automatiquement des dizaines de fois avec des utilisateurs différents de MovieLens, pour voir si notre modèle propose de bonnes recommandations dans ce scénario précis à chaque fois.

In [43]:
def recommander_films(id_utilisateur, nombre_films_choisis=4, k=10, seuil_appreciation=4.0):
    # On prend les films que cet utilisateur a bien notés dans l'entraînement
    notes_utilisateur = notes_entrainement[
        (notes_entrainement['userId'] == id_utilisateur) & 
        (notes_entrainement['rating'] >= seuil_appreciation)
    ]
    
    if len(notes_utilisateur) < nombre_films_choisis:
        return None, None  # pas assez de films aimés pour simuler ce cas
    
    # On tire au hasard "nombre_films_choisis" films parmi ceux qu'il a aimés
    # (comme s'il les avait lui-même choisis comme films de référence dans l'application)
    films_choisis = notes_utilisateur.sample(n=nombre_films_choisis, random_state=1)['movieId'].tolist()
    
    indices_notes = [position_film_combine[f] for f in films_choisis if f in position_film_combine]
    
    if len(indices_notes) == 0:
        return None, None
    
    # Profil moyen calculé sur SEULEMENT ces quelques films (comme dans la vraie application)
    profil_utilisateur = matrice_combinee[indices_notes].mean(axis=0)
    profil_utilisateur = np.asarray(profil_utilisateur)
    
    scores = cosine_similarity(profil_utilisateur, matrice_combinee)[0]
    
    for idx in indices_notes:
        scores[idx] = -1
    
    meilleurs_indices = np.argsort(scores)[::-1][:k]
    films_recommandes = [films['movieId'].iloc[i] for i in meilleurs_indices]
    
    # On vérifie si ces recommandations correspondent à d'autres films qu'il a aimés (dans le test)
    notes_test_utilisateur = notes_test[notes_test['userId'] == id_utilisateur]
    films_aimes_test = notes_test_utilisateur[notes_test_utilisateur['rating'] >= seuil_appreciation]['movieId'].tolist()
    
    if len(films_aimes_test) == 0:
        return None, None
    
    nb_corrects = len(set(films_recommandes) & set(films_aimes_test))
    precision = nb_corrects / k
    recall = nb_corrects / len(films_aimes_test)
    
    return precision, recall

In [44]:
# Tester sur l'échantillon de 100 utilisateurs 

precisions_peu_films = []
recalls_peu_films = []

for id_utilisateur in echantillon_utilisateurs:
    precision, recall = recommander_films(id_utilisateur, nombre_films_choisis=4, k=10)
    if precision is not None:
        precisions_peu_films.append(precision)
        recalls_peu_films.append(recall)

print(f"Precision@10 moyenne (profil de 4 films) sur {len(precisions_peu_films)} utilisateurs : {np.mean(precisions_peu_films):.3f}")
print(f"Recall@10 moyen (profil de 4 films) sur {len(precisions_peu_films)} utilisateurs : {np.mean(recalls_peu_films):.3f}")


Precision@10 moyenne (profil de 4 films) sur 96 utilisateurs : 0.010
Recall@10 moyen (profil de 4 films) sur 96 utilisateurs : 0.012


Commentaire : En simulant les conditions réelles de l'application (un utilisateur sélectionnant 4 films appréciés, plutôt qu'un historique complet de notes), le modèle content-based enrichi obtient une Precision@10 de 0,010 et un Recall@10 de 0,012, soit une amélioration nette par rapport au test précédent avec un profil moyen calculé sur un grand nombre de films (0,002). Ce résultat confirme que le problème de dilution du profil moyen expliquait une partie de la faible performance observée, sans totalement la résoudre. Le score reste néanmoins largement inférieur à celui du filtrage collaboratif (0,130), qui bénéficie d'un signal de recommandation plus fort en exploitant directement les comportements réels d'autres utilisateurs, un signal que le content-based, quelle que soit la richesse de son contenu, ne peut pas reproduire. Le content-based conserve cependant un avantage complémentaire majeur pour l'application finale : il reste utilisable pour un nouvel utilisateur sans aucun historique de notes, contrairement au filtrage collaboratif qui nécessite des données de comportement pour fonctionner (problème du "cold start").

Étape 10 : Conclusion 

Ce notebook avait pour objectif de tester si l'enrichissement du contenu des films (synopsis, acteurs, réalisateur, en plus des genres) permettait d'améliorer significativement les recommandations par rapport au modèle genres seuls du notebook n°2. Les résultats montrent que l'enrichissement améliore la cohérence qualitative des recommandations pour un film isolé (les suites de franchises comme Toy Story et Spider-Man remontent naturellement en tête), mais n'améliore que très légèrement les scores chiffrés de Precision@k et Recall@k, qui restent largement inférieurs à ceux du filtrage collaboratif. Le tableau ci-dessous résume les résultats obtenus sur les trois approches testées jusqu'ici.

4 Modèles     	        Ce qu'il utilise	        Precision@10	        Recall@10
----------------------------------------------------------------------------------------
Modèle genres seuls	    Genre uniquement	            0,003	                0,004


Modèle enrichi (profil sur 
historique complet)	    Genre + synopsis + 
                        acteurs + réalisateur	        0,002	                0,002

                    
Modèle enrichi (profil 
sur 4 films, conditions 
réelles)	            Genre + synopsis + 
                        acteurs + réalisateur	        0,010	                0,012


Filtrage collaboratif	Comportement des utilisateurs	0,130	                0,174


Ces résultats orientent la stratégie retenue pour l'application finale : le filtrage collaboratif sera utilisé comme moteur principal de recommandation dès qu'un utilisateur dispose d'un historique suffisant, tandis que le modèle content-based enrichi sera utilisé pour les nouveaux utilisateurs sans historique (problème du "cold start"), ainsi que pour la fonctionnalité de recherche de films similaires à un film donné.

In [45]:
# Sauvegarder le modèle enrichi pour l'utiliser dans l'application Streamlit

import pickle

# On sauvegarde la matrice combinée et la matrice de similarité,
# pour les réutiliser plus tard sans tout recalculer (notamment dans l'application Streamlit)
with open('/Users/nazmanazirhussain/Desktop/RecommandationFilm/models/matrice_combinee.pkl', 'wb') as fichier:
    pickle.dump(matrice_combinee, fichier)

with open('/Users/nazmanazirhussain/Desktop/RecommandationFilm/models/similarite_combinee.pkl', 'wb') as fichier:
    pickle.dump(similarite_combinee, fichier)

print("Modèle enrichi sauvegardé avec succès")

Modèle enrichi sauvegardé avec succès
